# 🏦 Банковский счёт: `Account` и `CreditAccount`

> **Цель:** реализовать класс `Account`, моделирующий банковский счёт, с историей операций и аналитикой.  
> Затем создать наследника `CreditAccount`.

---

## 📌 1. Класс `Account`

### Инициализация

```python
Account(account_holder, balance=0)
```

| Параметр | Тип | По умолчанию | Описание |
|----------|-----|--------------|----------|
| `account_holder` | `str` | — | Имя владельца счёта |
| `balance` | `float` | `0` | Начальный баланс. Не может быть отрицательным |

### Атрибуты

| Атрибут | Тип | Описание |
|---------|-----|----------|
| `holder` | `str` | Имя владельца |
| `_balance` | `float` | Приватный текущий баланс |
| `operations_history` | `list` / `tuple` | История операций |

> ⚠️ **Важно:** каждая операция хранится как структурированная запись — словарь или кортеж.  
> Минимальный набор полей:
> - тип операции: `'deposit'` или `'withdraw'`;
> - сумма;
> - дата и время;
> - текущий баланс после операции;
> - статус: `'success'` или `'fail'`.

#### Пример записи операции

```python
{
    "type": "deposit",
    "amount": 1000.0,
    "datetime": datetime.now(),
    "balance_after": 1000.0,
    "status": "success"
}
```

---

## 🛠️ 2. Методы `Account`

### `__init__(self, account_holder, balance=0)`
Конструктор класса.

### `deposit(self, amount)`
Пополнение счёта.

- принимает сумму;
- сумма должна быть положительной;
- при успехе обновляет баланс и добавляет запись в историю.

### `withdraw(self, amount)`
Снятие средств.

- принимает сумму;
- сумма должна быть положительной;
- проверяет, достаточно ли средств;
- если средств недостаточно — операция не проходит, но попытка фиксируется в истории со статусом `'fail'`;
- при успехе обновляет баланс и добавляет запись.

### `get_balance(self)`
Возвращает текущий баланс.

### `get_history(self)`
Возвращает историю операций.

> 💡 **Подсказка:** для работы с датой и временем используйте `datetime.now()` из модуля `datetime`.

---

## 💳 3. Класс `CreditAccount(Account)`

Наследует всю функциональность `Account` и добавляет кредитную логику.

### Особенности

- При инициализации принимает дополнительный параметр `credit_limit`.
- Баланс может быть отрицательным, но не ниже `-credit_limit`.
- По запросу показывает доступные кредитные средства:  
  `доступно = текущий баланс + credit_limit`
- В историю операций добавляется информация о том, были ли использованы кредитные средства.

### Инициализация

```python
CreditAccount(account_holder, balance=0, credit_limit=0)
```

---

## ✅ Чек-лист реализации

- [ ] Создан класс `Account`.
- [ ] Реализован конструктор с проверкой баланса.
- [ ] Реализованы `deposit`, `withdraw`, `get_balance`, `get_history`.
- [ ] История операций хранит структурированные записи.
- [ ] Неудачные попытки снятия фиксируются.
- [ ] Создан класс `CreditAccount`.
- [ ] Учтён кредитный лимит.
- [ ] Добавлена информация об использовании кредитных средств.

---

## 📚 Полезные импорты

```python
from datetime import datetime
```

---

> **Совет:** для красивого вывода истории можно вернуть список словарей или преобразовать его в `pandas.DataFrame`.

# 📊 Критерии оценивания проекта

> **Максимум по базовым критериям: 8 баллов** — это оценка **«отлично»**.  
> 💡 Преподаватель может добавить **1–2 балла** и поставить **9 или 10**, если работа выполнена сверх ожиданий и выходит за рамки программы.

---

| № | Критерий | Баллы | Описание |
|:--:|:---------|:-----:|:---------|
| 1 | 🧱 **Корректность ООП** | **3** | Класс корректно инициализируется, используются инкапсуляция (приватные атрибуты, геттеры), методы `deposit` и `withdraw` работают без ошибок, логически верно обрабатывают неверные входные данные (отрицательные суммы) |
| 2 | 📜 **История операций** | **2** | История сохраняется в структурированном виде. Метод `get_history` возвращает информацию в удобном формате |
| 3 | 🕒 **Работа с датами** | **1** | В каждой операции корректно фиксируется дата и время её проведения с помощью модуля `datetime` |
| 4 | 🧬 **Наследование** | **2** | Корректно реализована функциональность класса `CreditAccount(Account)`: в реализации учтены все заданные особенности кредитного счёта |
| 5 | ✨ **Чистота кода** | **−1** за каждое нарушение | Код откомментирован, читаемый, соответствует PEP8 |

---

> **Итого:** `3 + 2 + 1 + 2 = 8` баллов — оценка **«отлично»**.  
> 🚀 Дополнительные баллы: **9–10**, если работа выходит за рамки программы.

In [ ]:
from datetime import datetime, timezone


class Account:
    """Моделирует банковский счёт и хранит историю операций."""

    def __init__(self, account_holder: str, balance: float = 0.0) -> None:
        """Создаёт банковский счёт с указанным владельцем и начальным балансом."""

        self.holder: str = account_holder

        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")

        self._balance: float = balance
        self._operations_history: list[dict[str, object]] = []

    def deposit(self, amount: float) -> bool:
        """Пополняет счет на положительную сумму."""

        if amount <= 0:
            raise ValueError("Сумма пополнения должна быть положительной")

        self._balance += amount
        status = "success"
        self._add_history(
            operation_type="deposit",
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=status,
        )
        return True

    def withdraw(self, amount: float) -> bool:
        """Снимает средства при наличии достаточной суммы на счёте."""

        if amount <= 0:
            raise ValueError("Сумма вывода наличных должна быть положительной")

        if amount > self._balance:
            status = "fail"
        else:
            self._balance -= amount
            status = "success"
        self._add_history(
            operation_type="withdraw",
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=status,
        )
        return status == "success"

    def get_balance(self) -> float:
        """Возвращает текущий баланс счёта."""

        return self._balance

    def _add_history(
        self,
        operation_type: str,
        amount: float,
        date_time: datetime,
        balance_after: float,
        status: str,
        credit_used: bool | None = None,
    ) -> None:
        """Добавляет запись об операции в историю счёта."""

        record: dict[str, object] = {
            "type": operation_type,
            "amount": amount,
            "datetime": date_time,
            "balance_after": balance_after,
            "status": status,
        }

        if credit_used is not None:
            record["credit_used"] = credit_used
        self._operations_history.append(record)

    def get_history(self) -> list[dict[str, object]]:
        """Возвращает историю операций по счету."""

        return [operation.copy() for operation in self._operations_history]
    
    def __str__(self) -> str:
        """Возвращает строковое представление банковского счёта."""
        
        return (
                f"Владелец: {self.holder}. "
                f"Баланс: {self._balance:.2f}"
            )


class CreditAccount(Account):
    """Моделирует банковский счёт с кредитным лимитом."""

    def __init__(
        self, account_holder: str, balance: float = 0.0, credit_limit: float = 0.0
    ) -> None:
        """Создаёт банковский счёт с указанным начальным балансом и кредитным лемитом"""

        super().__init__(account_holder, balance)
        if credit_limit < 0:
            raise ValueError("Кредитный лимит не может быть отрицательным")
        self._credit_limit = credit_limit

    def withdraw(self, amount: float) -> bool:
        """Снимает средства с учётом установленного кредитного лимита."""

        available_funds = self.get_available_funds()

        if amount <= 0:
            raise ValueError("Суммая вывода наличных должна быть положительной")

        if amount > available_funds:
            status = "fail"
        else:
            self._balance -= amount
            status = "success"

        credit_used = status == "success" and self._balance < 0
        self._add_history(
            operation_type="withdraw",
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=status,
            credit_used=credit_used,
        )

        return status == "success"

    def get_available_funds(self) -> float:
        """Возвращает общую сумму доступных собственных и кредитных средств."""

        return self._balance + self._credit_limit

    def get_credit_limit(self) -> float:
        """Возвращает кредитный лимит."""

        return self._credit_limit
    
    def __str__(self) -> str:
        """Возвращает строковое представление кредитного счёта."""

        available_funds = self.get_available_funds()

        return (
            f"Владелец: {self.holder}. "
            f"Баланс: {self._balance:.2f}. "
            f"Кредитный лимит: {self._credit_limit:.2f}. "
            f"Доступно средств: {available_funds:.2f}"
        )

In [ ]:
def test_account_creation() -> None:
    """Проверяет создание обычного банковского счёта."""
    
    account = Account("Илья Кижватов", 1000)
    
    assert account.holder == "Илья Кижватов", "Неверно сохранено имя владельца"
    assert account.get_balance() == 1000, "Неверный начальный баланс"
    assert account.get_history() == [], "История нового счёта должна быть пустой"

def test_account_default_balance() -> None:
    """Проверяет возможность создать счет, без передачи параметра начальный баланс."""
    
    account = Account("Анна")
    
    assert account.holder == "Анна", "Неверно сохранено имя владельца"
    assert account.get_balance() == 0, "Неверный начальный баланс, должен быть равен 0.0"
    assert account.get_history() == [], "История нового счёта должна быть пустой"
    
def test_successful_deposit() -> None:
    """Проверяет успешное пополнение счёта."""
    
    account = Account("Илья", 1000)
    result = account.deposit(500.3)
    history = account.get_history()
    
    assert account.get_balance() == 1500.3, "Неверный баланс после пополнения."
    assert result is True, "Функция deposit() должна была вернуть True"
    
    assert len(history) == 1, "Неверное количество записей в истории операции по счету"
    operation = history[0]
    assert operation["type"] == "deposit", "Неверный тип операции, при пополнении баланса"
    assert operation["amount"] == 500.3, "Сумма операции должна быть равна 500.3"
    assert operation["balance_after"] == 1500.3, "Неверный баланс после пополнения счёта"
    assert operation["status"] == "success", "Неверный статус операции"
    
def test_multiple_deposits() -> None:
    """Проверяет несколько последовательных пополнений счёта."""
    
    account = Account("Илья", 1000)
    first_deposit = account.deposit(500.3)
    second_deposit = account.deposit(100.3)
    
    history = account.get_history()
    
    assert account.get_balance() == 1600.6, "Неверный баланс после пополнения."
    assert first_deposit is True, "Результат первого пополнения должен быть True"
    assert second_deposit is True, "Результат второго пополнения должен быть True"
    
    assert len(history) == 2, "В истории должно быть две операции"
    operation = history[0]
    assert operation["type"] == "deposit", "Неверный тип операции, при пополнении баланса"
    assert operation["amount"] == 500.3, "Сумма операции должна быть равна 500.3"
    assert operation["balance_after"] == 1500.3, "Неверный баланс после пополнения счёта"
    assert operation["status"] == "success", "Неверный статус операции"
    
    operation = history[1]
    assert operation["type"] == "deposit", "Неверный тип операции, при пополнении баланса"
    assert operation["amount"] == 100.3, "Сумма операции должна быть равна 100.3"
    assert operation["balance_after"] == 1600.6, "Неверный баланс после пополнения счёта"
    assert operation["status"] == "success", "Неверный статус операции"

def test_negative_deposit() -> None:
    """Проверяет запрет отрицательной суммы пополнения."""

    account = Account("Илья", 1000)

    try:
        account.deposit(-500)
    except ValueError:
        pass
    else:
        raise AssertionError(
            "deposit() должен вызвать ValueError"
        )

    assert account.get_balance() == 1000, "При ошибочном пополнении изменился баланс счёта"
    assert account.get_history() == [], "При ошибочном пополнении изменилась история операций"
    
def test_zero_deposit() -> None:
    """Проверяет целостность данных при вызове функции пополнения с отрицательным значением"""
    
    account = Account("Илья", 1000)
    
    try:
        account.deposit(0)
    except ValueError:
        pass
    else:
        raise AssertionError(
            "deposit() должен вызвать ValueErorr"
        )
        
    assert account.get_balance() == 1000, "При ошибочном пополнении изменился баланс счёта"
    assert account.get_history() == [], "При ошибочном пополнении изменилась история операций"
    
def test_successful_withdrawal() -> None:
    """Проверяет успешное снятие средств."""

    # Arrange
    account = Account("Илья", 1000)

    # Act
    result = account.withdraw(300)
    history = account.get_history()

    # Assert
    assert result is True, "Успешное снятие собственных средств должно вернуть True"
    assert account.get_balance() == 700, "Неверный баланс после снятия наличных"
    assert len(history) == 1, "Неверное количество операций в истории счёта"

    operation = history[0]

    assert operation["type"] == "withdraw", "Неверный тип операции в истории, при снятии наличных"
    assert operation["amount"] == 300, "Неверная сумма операции в истории при снятии наличных"
    assert operation["balance_after"] == 700, "Неверный баланс в истории после снятия наличных"
    assert operation["status"] == "success", "Неверный статус операции в истории при успешном снятии наличных"
    
def test_withdrawal_of_entire_balance() -> None:
    """Проверяет снятие всей доступной суммы."""

    # Arrange
    account = Account("Илья", 1000)

    # Act
    result = account.withdraw(1000)
    history = account.get_history()

    # Assert
    assert result is True, "Успешное снятие собственных средств должно вернуть True"
    assert account.get_balance() == 0, "Баланс не равен нулю, при снятии всей суммы"
    assert len(history) == 1, "Неверное количество записей в истории операций по счету"
    assert history[0]["status"] == "success", "Неверный статус операции при снятии наличных"
    assert history[0]["balance_after"] == 0, "Неверный баланс после снятие наличных в истории операций"

def test_withdrawal_with_insufficient_funds() -> None:
    """Проверяет отказ при недостатке средств."""

    # Arrange
    account = Account("Илья", 1000)

    # Act
    result = account.withdraw(1500)
    history = account.get_history()

    # Assert
    assert result is False, "Запрещенная операция станятие наличных вернула True"
    assert account.get_balance() == 1000, "После не выполненной операции снятия, изменился баланс"
    assert len(history) == 1, "Не выполненная операции снятия не записалась в историю операций"

    operation = history[0]

    assert operation["type"] == "withdraw", "Неверный тип операции в истории"
    assert operation["amount"] == 1500, "Неверная сумма записана в истории при выполнении операции"
    assert operation["balance_after"] == 1000, "Неверный баланс после не выполненной операции снятия"
    assert operation["status"] == "fail", "Неверный статус операции"

def test_negative_withdrawal() -> None:
    """Проверяет запрет отрицательной суммы снятия."""

    # Arrange
    account = Account("Илья", 1000)

    # Act и Assert
    try:
        account.withdraw(-300)
    except ValueError:
        pass
    else:
        raise AssertionError(
            "withdraw() должен вызвать ValueError "
            "для отрицательной суммы"
        )

    assert account.get_balance() == 1000, "Изменился баланс после запрещенной операции снятия"
    assert account.get_history() == [], "В историю записалась операция которой не было"
    
def test_zero_withdrawal() -> None:
    """Проверяет запрет отрицательной суммы снятия."""

    # Arrange
    account = Account("Илья", 1000)

    # Act и Assert
    try:
        account.withdraw(0)
    except ValueError:
        pass
    else:
        raise AssertionError(
            "withdraw() должен вызвать ValueError "
            "для нулевой суммы"
        )

    assert account.get_balance() == 1000, "Изменился баланс после запрещенной операции снятия"
    assert account.get_history() == [], "В историю записалась операция которой не было"
    
def test_multiple_withdrawals() -> None:
    """Проверяет отказ при недостатке средств."""

    # Arrange
    account = Account("Илья", 1000)

    # Act
    first_withdrawals = account.withdraw(500)
    second_withdrawals = account.withdraw(200)
    history = account.get_history()

    # Assert
    assert first_withdrawals is True, "Неверный результат выполнения первой операция снятия"
    assert second_withdrawals is True, "Неверный результат выполнения второй операция снятия"
    
    assert account.get_balance() == 300, "Неверный баланс после выполнения двух снятий"
    assert len(history) == 2, "Неверное количество операций в истории по счету после выполнения двух снятий"

    operation0 = history[0]

    assert operation0["type"] == "withdraw", "Неверный тип операции в истории"
    assert operation0["amount"] == 500, "Неверная сумма записана в истории при выполнении операции"
    assert operation0["balance_after"] == 500, "Неверный баланс после выполненной операции снятия"
    assert operation0["status"] == "success", "Неверный статус операции"
    
    operation1 = history[1]
    
    assert operation1["type"] == "withdraw", "Неверный тип операции в истории"
    assert operation1["amount"] == 200, "Неверная сумма записана в истории при выполнении операции"
    assert operation1["balance_after"] == 300, "Неверный баланс после выполненной операции снятия"
    assert operation1["status"] == "success", "Неверный статус операции"
    
def test_history_record_structure() -> None:
    """Проверяет структуру записи обычного счёта."""

    # Arrange
    account = Account("Илья", 1000)

    # Act
    account.deposit(500)
    operation = account.get_history()[0]

    # Assert
    expected_keys = {
        "type",
        "amount",
        "datetime",
        "balance_after",
        "status",
    }

    assert set(operation.keys()) == expected_keys, "Неверные ключи словаря записи об операции"

def test_history_datetime() -> None:
    """Проверяет тип даты и наличие часового пояса."""

    # Arrange
    account = Account("Илья", 1000)

    # Act
    account.deposit(500)
    operation_time = account.get_history()[0]["datetime"]

    # Assert
    assert isinstance(operation_time, datetime), "Время операции должно быть объектом datetime"
    assert operation_time.tzinfo is not None, "Время операции должно содержать часовой пояс"
    assert operation_time.utcoffset() is not None, "Часовой пояс операции должен иметь смещение от UTC"
    
def test_history_operation_time() -> None:
    """Проверяет время создания записи об операции."""

    # Arrange
    account = Account("Илья", 1000)
    time_before = datetime.now(timezone.utc)

    # Act
    account.deposit(500)
    time_after = datetime.now(timezone.utc)

    operation_time = account.get_history()[0]["datetime"]

    # Assert
    assert isinstance(operation_time, datetime), "Время операции должно быть объектом datetime"
    assert time_before <= operation_time <= time_after, "Время операции должно находиться между началом и окончанием вызова"

def test_history_operation_order() -> None:
    """Проверяет порядок операций в истории."""

    # Arrange
    account = Account("Илья", 1000)

    # Act
    account.deposit(500)
    account.withdraw(200)
    account.deposit(100)

    history = account.get_history()

    # Assert
    assert len(history) == 3, "В истории должно быть три операции"

    assert history[0]["type"] == "deposit", "Первая операция должна быть пополнением"
    assert history[0]["amount"] == 500, "Сумма первой операции должна быть равна 500"
    assert history[0]["balance_after"] == 1500, "После первой операции баланс должен быть равен 1500"

    assert history[1]["type"] == "withdraw", "Вторая операция должна быть снятием"
    assert history[1]["amount"] == 200, "Сумма второй операции должна быть равна 200"
    assert history[1]["balance_after"] == 1300, "После второй операции баланс должен быть равен 1300"

    assert history[2]["type"] == "deposit", "Третья операция должна быть пополнением"
    assert history[2]["amount"] == 100, "Сумма третьей операции должна быть равна 100"
    assert history[2]["balance_after"] == 1400, "После третьей операции баланс должен быть равен 1400"

def test_history_list_is_protected() -> None:
    """Проверяет защиту списка истории от внешнего изменения."""

    # Arrange
    account = Account("Илья", 1000)
    account.deposit(500)

    # Act
    received_history = account.get_history()
    received_history.clear()

    actual_history = account.get_history()

    # Assert
    assert received_history == [], "Полученную копию истории должно быть возможно очистить"
    assert len(actual_history) == 1, "Очистка копии не должна изменять внутреннюю историю"

def test_history_records_are_protected() -> None:
    """Проверяет защиту отдельных записей истории."""

    # Arrange
    account = Account("Илья", 1000)
    account.deposit(500)

    # Act
    received_history = account.get_history()
    received_history[0]["amount"] = 999_999

    actual_history = account.get_history()

    # Assert
    assert received_history[0]["amount"] == 999_999, "Сумма в полученной копии должна измениться"
    assert actual_history[0]["amount"] == 500, "Изменение копии не должно изменять внутреннюю запись"
    
def test_credit_account_creation() -> None:
    """Проверяет создание кредитного счёта."""

    # Arrange и Act
    account = CreditAccount(
        account_holder="Илья",
        balance=1000,
        credit_limit=5000,
    )

    # Assert
    assert account.holder == "Илья", "Имя владельца должно быть сохранено"
    assert account.get_balance() == 1000, "Начальный баланс должен быть равен 1000"
    assert account.get_credit_limit() == 5000, "Кредитный лимит должен быть равен 5000"
    assert account.get_available_funds() == 6000, "Доступная сумма должна быть равна 6000"
    assert account.get_history() == [], "История нового кредитного счёта должна быть пустой"

def test_credit_account_default_values() -> None:
    """Проверяет параметры кредитного счёта по умолчанию."""

    # Arrange и Act
    account = CreditAccount("Анна")

    # Assert
    assert account.holder == "Анна", "Имя владельца должно быть сохранено"
    assert account.get_balance() == 0, "Баланс по умолчанию должен быть равен нулю"
    assert account.get_credit_limit() == 0, "Кредитный лимит по умолчанию должен быть равен нулю"
    assert account.get_available_funds() == 0, "Доступная сумма должна быть равна нулю"
    assert account.get_history() == [], "История нового кредитного счёта должна быть пустой"
    
def test_negative_credit_limit() -> None:
    """Проверяет запрет отрицательного кредитного лимита."""

    try:
        CreditAccount(
            account_holder="Илья",
            balance=1000,
            credit_limit=-5000,
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Отрицательный кредитный лимит должен "
            "вызывать ValueError"
        )
        

def test_withdraw_own_funds_from_credit_account() -> None:
    """Проверяет снятие без использования кредитных средств."""

    # Arrange
    account = CreditAccount(
        account_holder="Илья",
        balance=1000,
        credit_limit=5000,
    )

    # Act
    result = account.withdraw(500)
    history = account.get_history()

    # Assert
    assert result is True, "Успешное снятие собственных средств должно вернуть True"
    assert account.get_balance() == 500, "После снятия 500 баланс должен быть равен 500"
    assert account.get_available_funds() == 5500, "После снятия доступная сумма должна быть равна 5500"
    assert len(history) == 1, "В истории должна быть одна операция"

    operation = history[0]

    assert operation["type"] == "withdraw", "Тип операции должен быть withdraw"
    assert operation["amount"] == 500, "Сумма операции должна быть равна 500"
    assert operation["balance_after"] == 500, "Баланс после операции должен быть равен 500"
    assert operation["status"] == "success", "Статус успешной операции должен быть success"
    assert operation["credit_used"] is False, "При снятии собственных средств кредит не должен использоваться"

def test_withdraw_credit_funds() -> None:
    """Проверяет снятие с использованием кредитных средств."""

    # Arrange
    account = CreditAccount(
        account_holder="Илья",
        balance=1000,
        credit_limit=5000,
    )

    # Act
    result = account.withdraw(3000)
    history = account.get_history()

    # Assert
    assert result is True, "Снятие в пределах кредитного лимита должно вернуть True"
    assert account.get_balance() == -2000, "После снятия баланс должен быть равен -2000"
    assert account.get_available_funds() == 3000, "Оставшаяся доступная сумма должна быть равна 3000"
    assert len(history) == 1, "В истории должна быть одна операция"

    operation = history[0]

    assert operation["type"] == "withdraw", "Тип операции должен быть withdraw"
    assert operation["amount"] == 3000, "Сумма операции должна быть равна 3000"
    assert operation["balance_after"] == -2000, "Баланс после операции должен быть равен -2000"
    assert operation["status"] == "success", "Статус успешной операции должен быть success"
    assert operation["credit_used"] is True, "При отрицательном балансе должно фиксироваться использование кредита"

def test_withdraw_all_available_funds() -> None:
    """Проверяет снятие всей доступной суммы."""

    # Arrange
    account = CreditAccount(
        account_holder="Илья",
        balance=1000,
        credit_limit=5000,
    )

    # Act
    result = account.withdraw(6000)
    history = account.get_history()

    # Assert
    assert result is True, "Снятие всей доступной суммы должно вернуть True"
    assert account.get_balance() == -5000, "Баланс должен быть равен отрицательному кредитному лимиту"
    assert account.get_available_funds() == 0, "После снятия всей суммы доступные средства должны быть равны нулю"
    assert len(history) == 1, "В истории должна быть одна операция"

    operation = history[0]

    assert operation["status"] == "success", "Статус операции должен быть success"
    assert operation["balance_after"] == -5000, "Баланс в истории должен быть равен -5000"
    assert operation["credit_used"] is True, "При снятии всей доступной суммы должен использоваться кредит"

def test_credit_limit_exceeded() -> None:
    """Проверяет отказ при превышении кредитного лимита."""

    # Arrange
    account = CreditAccount(
        account_holder="Илья",
        balance=1000,
        credit_limit=5000,
    )

    # Act
    result = account.withdraw(6001)
    history = account.get_history()

    # Assert
    assert result is False, "Снятие сверх доступной суммы должно вернуть False"
    assert account.get_balance() == 1000, "Неудачная операция не должна изменять баланс"
    assert account.get_available_funds() == 6000, "Неудачная операция не должна изменять доступную сумму"
    assert len(history) == 1, "Неудачная попытка должна быть записана в историю"

    operation = history[0]

    assert operation["type"] == "withdraw", "Тип операции должен быть withdraw"
    assert operation["amount"] == 6001, "В истории должна храниться запрошенная сумма 6001"
    assert operation["balance_after"] == 1000, "Баланс после неудачной операции должен остаться равным 1000"
    assert operation["status"] == "fail", "Статус неудачной операции должен быть fail"
    assert operation["credit_used"] is False, "При неудачной операции кредит не должен использоваться"
    
def test_negative_credit_withdrawal() -> None:
    """Проверяет запрет отрицательной суммы снятия."""

    # Arrange
    account = CreditAccount(
        account_holder="Илья",
        balance=1000,
        credit_limit=5000,
    )

    # Act и Assert
    try:
        account.withdraw(-100)
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Отрицательная сумма снятия должна "
            "вызывать ValueError"
        )

    assert account.get_balance() == 1000, "Ошибочный вызов не должен изменять баланс"
    assert account.get_available_funds() == 6000, "Ошибочный вызов не должен изменять доступную сумму"
    assert account.get_history() == [], "Ошибочный вызов не должен добавляться в историю"
    
def test_zero_credit_withdrawal() -> None:
    """Проверяет запрет нулевой суммы снятия."""
    
    account = CreditAccount(
            account_holder="Илья",
            balance=1000,
            credit_limit=5000,
        )
    try:
        account.withdraw(0)
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Нулевая сумма снятия должна вызывать ValueError"
        )
    assert account.get_balance() == 1000, "Ошибочный вызов не должен изменять баланс"
    assert account.get_available_funds() == 6000, "Ошибочный вызов не должен изменять доступную сумму"
    assert account.get_history() == [], "Ошибочный вызов не должен добавляться в историю"
        
def test_multiple_credit_withdrawals() -> None:
    """Проверяет несколько снятий с кредитного счёта."""

    # Arrange
    account = CreditAccount(
        account_holder="Илья",
        balance=1000,
        credit_limit=5000,
    )

    # Act
    first_result = account.withdraw(500)
    second_result = account.withdraw(1000)
    third_result = account.withdraw(5000)

    history = account.get_history()

    # Assert
    assert first_result is True, "Первое снятие должно вернуть True"
    assert second_result is True, "Второе снятие должно вернуть True"
    assert third_result is False, "Снятие сверх доступной суммы должно вернуть False"

    assert account.get_balance() == -500, "Итоговый баланс должен быть равен -500"
    assert account.get_available_funds() == 4500, "Итоговая доступная сумма должна быть равна 4500"
    assert len(history) == 3, "В истории должно быть три операции"

    first_operation = history[0]
    second_operation = history[1]
    third_operation = history[2]

    assert first_operation["status"] == "success", "Первая операция должна быть успешной"
    assert first_operation["balance_after"] == 500, "После первой операции баланс должен быть равен 500"
    assert first_operation["credit_used"] is False, "Первая операция не должна использовать кредит"
    assert second_operation["status"] == "success", "Вторая операция должна быть успешной"
    assert second_operation["balance_after"] == -500, "После второй операции баланс должен быть равен -500"
    assert second_operation["credit_used"] is True, "Вторая операция должна использовать кредит"

    assert third_operation["status"] == "fail", "Третья операция должна быть отклонена"
    assert third_operation["balance_after"] == -500, "Неудачная операция не должна изменять баланс"
    assert third_operation["credit_used"] is False, "Неудачная операция не должна использовать кредит"

